In [ ]:
!pip install "unstructured[pdf]"

In [ ]:
!pip install pypdf


In [ ]:
from langchain_community.document_loaders import DirectoryLoader

pdfs = DirectoryLoader("documentos",glob="*.pdf").load()

len(pdfs)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [ ]:
from langchain_text_splitters import CharacterTextSplitter 

splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=1250, chunk_overlap=250
)

pedacos = splitter.split_documents(pdfs)
len(pedacos)

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="bge-m3:567m",
    base_url="http://localhost:11434"
    )

vector_store = FAISS.from_documents(
    documents=pedacos,embedding=embeddings
)

In [ ]:
retriever = vector_store.as_retriever()

from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser



prompt = ChatPromptTemplate.from_messages(
    [
        ("system","Responda usando exclusivamente os conteúdo fornecido. \n\nContexto:\n{Contexto}"),
        ("human","{query}")
    ]
)

modelo = OllamaLLM(
    model="gemma3:4b",
    base_url="http://localhost:11434"    
)

cadeia= prompt|modelo|StrOutputParser


In [ ]:
pergunta = "como fazer um seguro de viagem?"
contexto_exemplo = "O seguro de viagem pode ser feito online em sites de seguradoras..."

# Você precisa passar EXATAMENTE as chaves 'query' e 'Contexto'
cadeia.invoke({
    "query": pergunta,
    "contexto": contexto_exemplo
})

In [ ]:
pergunta = "como fazer um seguro de viagem?"

trechos = retriever.invoke(pergunta)

contexto = "\n\n".join(trecho.page_content for trecho in trechos)

cadeia.invoke({"query": pergunta, "contexto": contexto})

In [ ]:
from langchain_core.runnables  import RunnablePassthrough

rag_chain =(
    {
        "contexto": "query": RunnablePassthrough() | retriever,
        "query": RunnablePassthrough()
    }
    | prompt | modelo | StrOutputParser()
)

In [ ]:
rag_chain.invoke(pergunta)

In [20]:
from langchain_core.runnables import RunnablePassthrough

# 1. Ajustei o Prompt para usar "contexto" minúsculo (padronização)
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda usando exclusivamente o conteúdo fornecido. \n\nContexto:\n{contexto}"),
        ("human", "{query}")
    ]
)

# 2. Função auxiliar para formatar os documentos vindos do retriever
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Construção da RAG Chain corrigida
rag_chain = (
    {
        "contexto": retriever | format_docs, # O retriever busca e o format_docs limpa o texto
        "query": RunnablePassthrough()       # A pergunta passa direto para cá
    }
    | prompt 
    | modelo 
    | StrOutputParser() # Com parênteses
)

# 4. Invocação
pergunta = "como fazer um seguro de viagem?"
resultado = rag_chain.invoke(pergunta)
print(resultado)

Para obter cobertura com o MasterAssist Plus, siga estes passos:

1.  **Emita o Bilhete de Seguro:** O documento é emitido através do portal www.aig.com/Mastercard/pt e deve ser apresentado em caso de sinistro. É imprescindível que o bilhete esteja em vigor, com validade de 12 meses a partir da data de emissão, e que a viagem tenha ocorrido durante esse período.
2.  **Notifique a Seguradora:** Assim que um incidente ou sinistro ocorrer, notifique a Seguradora o mais rápido possível.
3.  **Preencha o Formulário de Sinistro:** A Seguradora fornecerá o formulário de sinistro. Preencha-o integralmente, devidamente assinado e datado.
4.  **Envie as Informações Exigidas:** Anexe todos os documentos que comprovem a ocorrência do sinistro (comprovante de pagamento da passagem, relatórios policiais, laudos médicos, etc.).

**Informações adicionais:**

*   **Observações:**
    *   A cobertura máxima de despesas médicas é de USD 25.000 por pessoa.
    *   Não há limite para o número de viagens.
 

In [35]:
from langchain.evaluation.qa import QAEvalChain

eval_chain = QAEvalChain.from_llm(modelo)

def avaliar(perguntas_respostas,geracoes):
    #perguntas_respostas: query, answer
    #gerações:result
    avaliacoes = eval_chain(perguntas_respostas,geracoes)
    corretas = 0
    for i in enumerate(perguntas_respostas):
        corretas = corretas + (1 if avaliacoes[i]["results"].split("\n")[-1].split(":")[-1].strip()== "CORRECT" else 0)
    return corretas/len(perguntas_respostas)   

ModuleNotFoundError: No module named 'langchain.evaluation'

In [ ]:
from langchain.evaluation